In [19]:
from bs4 import BeautifulSoup
import requests, os, csv, urllib.parse, urllib3, time, random
from requests.adapters import HTTPAdapter
from requests.packages.urllib3.util.retry import Retry

import numpy as np
import pandas as pd
from tqdm import tqdm
import re,os,glob, time,string
urllib3.disable_warnings()

import web_general
from IPython.display import Image

import requests
from bs4 import BeautifulSoup
import time

In [20]:
# Load the data
df_sample = pd.read_csv('NPO_sample.csv')
df_sample

,Unnamed: 0,ID,Search,Name,Province,Reg_no,DOR,Status,DC,FM,Page
0,0,68599106,NaN,Reatlegile Day Care Centre,Gauteng,247-285 NPO,30/07/2020,Registered,,,10
1,1,68539213,NaN,Reel Science Co.,Western Cape,246-837 NPO,16/07/2020,Registered,,,10
2,2,55790,NaN,Regodile Community Development Organisation,Limpopo,055-790 NPO,02/07/2007,Registered,,,10
3,3,52290592,NaN,Restore Alcohol & Drug Recovery Centre Glenvista,Gauteng,216-181 NPO,02/10/2018,Registered,,,10
4,4,34673111,NaN,Reya Gona Special Needs Center,Tembisa,185-657 NPO,20/02/2017,Registered,,,10
5,5,50153123,NaN,RHEMA The Cross Ministries,Gauteng,210-934 NPO,28/06/2018,Registered,,,10
6,6,56221573,NaN,Rhinosaverz,Western Cape,225-622 NPO,04/04/2019,Registered,,,10
7,7,43099440,NaN,Rise And Shine Youth Skill Development,Hluhluwe,193-842 NPO,04/08/2017,Registered,,,10


In [21]:
id_list = df_sample['ID'].astype(str).values.tolist()
id_list

['68599106',
 '68539213',
 '55790',
 '52290592',
 '34673111',
 '50153123',
 '56221573',
 '43099440']

Enter the URL of the website you will like to webscrape

In [22]:
URL='https://www.npo.gov.za/PublicNpo/Npo/68599106'

In [23]:
soup=web_general.parse_listing(web_general.buildSearchUrl(URL), web_general.get_proxies())
if not soup:print("Not found")
else: print("good to go")

Not found


We can now webscrape everything that we can see in the soup file.<br>

In [24]:
base_url="https://www.npo.gov.za/PublicNpo/Npo/DetailsPublicDocs/"

In [26]:
# creating empty list
name_list = []
reg_no_list = []
cat123_list = []
NPOType_list = []
dor_list = []

for ID in id_list:
    search_url=base_url+ID
    soup=web_general.parse_listing(web_general.buildSearchUrl(search_url), web_general.get_proxies())
    #passing through all web scrapped files to a list
    name_list.append(soup.find("h1", class_="widget-head-border colr").text.split("\n")[1])
    reg_no_list.append(soup.find("div", class_="padbox").text.split("\n")[7])
    cat123_list.append(soup.find("div", class_="padbox").text.split("\n")[8])
    NPOType_list.append(soup.find("div", class_="padbox").text.split("\n")[10])
    dor_list.append(soup.find("div", class_="padbox").text.split("\n")[13])

In [27]:
cat123_list

['Social Services                                                        > Services to Children > ECD and Partial Care Centres\r',
 'Education and Research                                                 > Research                                                               > Science and technoloy                                                 \r',
 'Health                                                                 > Hospitals and rehabilitation                                           > Rehabilitation                                                        \r',
 'Law, Advocacy, and Politics                                            > Law and Legal Services                                                 > Rehabilitation of offenders                                           \r',
 'Social Services                                                        > Services to Children > Homes for Children with Special Needs\r',
 'Religion                                                  

In [28]:
# create df and pass in all lists created above
pd.set_option('display.max_colwidth', None)
df = pd.DataFrame(np.column_stack([id_list, name_list, reg_no_list, cat123_list, NPOType_list, dor_list ]),columns=["ID","Name","Reg_no","Cat123","NPO_Type","DOR"])
df

,ID,Name,Reg_no,Cat123,NPO_Type,DOR
0,68599106,NPO Details: Reatlegile Day Care Centre,Registration No.: 247-285 NPO,Social Services > Services to Children > ECD and Partial Care Centres\r,Voluntary Association,30/07/2020\r
1,68539213,NPO Details: Reel Science Co.,Registration No.: 246-837 NPO,Education and Research > Research > Science and technoloy \r,Not for Profit Companies,16/07/2020\r
2,55790,NPO Details: Regodile Community Development Organisation,Registration No.: 055-790 NPO,Health > Hospitals and rehabilitation > Rehabilitation \r,Voluntary Association,02/07/2007\r
3,52290592,NPO Details: Restore Alcohol & Drug Recovery Centre Glenvista,Registration No.: 216-181 NPO,"Law, Advocacy, and Politics > Law and Legal Services > Rehabilitation of offenders \r",Not for Profit Companies,02/10/2018\r
4,34673111,NPO Details: Reya Gona Special Needs Center,Registration No.: 185-657 NPO,Social Services > Services to Children > Homes for Children with Special Needs\r,Voluntary Association,20/02/2017\r
5,50153123,NPO Details: RHEMA The Cross Ministries,Registration No.: 210-934 NPO,Religion > Religious Congregations and Associations > Congregations \r,Not for Profit Companies,28/06/2018\r
6,56221573,NPO Details: Rhinosaverz,Registration No.: 225-622 NPO,"Development and Housing > Economic, Social and Community Development > Social Development \r",Not for Profit Companies,04/04/2019\r
7,43099440,NPO Details: Rise And Shine Youth Skill Development,Registration No.: 193-842 NPO,"Development and Housing > Economic, Social and Community Development > Community and neighborhood organisations \r",Voluntary Association,04/08/2017\r


In [29]:
##Verifying datatypes for join columns before preproccessing for each column
print("#################### webscrapped part 1############################ ")
print(df_sample.info())
print("")
print("####################WEB scrapped part 2############################ ")
print(df.info())

#################### webscrapped part 1############################ 
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  8 non-null      int64  
 1   ID          8 non-null      int64  
 2   Search      0 non-null      float64
 3   Name        8 non-null      object 
 4   Province    8 non-null      object 
 5   Reg_no      8 non-null      object 
 6   DOR         8 non-null      object 
 7   Status      8 non-null      object 
 8   DC          8 non-null      object 
 9   FM          8 non-null      object 
 10  Page        8 non-null      int64  
dtypes: float64(1), int64(3), object(7)
memory usage: 832.0+ bytes
None

####################WEB scrapped part 2############################ 
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --

In [34]:
def pre_pocessing(df,df2):
    #spliting 
    df[["cat1", "cat2", "cat3"]] = df['Cat123'].str.split(pat='>',expand=True)
    #removing all uncessary words in each column
    df['cat3'] = df['cat3'].str.replace("\r", "", regex=True)
    df.drop(["Cat123"], axis=1, inplace= True)
    df['DOR'] = df['DOR'].str.replace("\r", "", regex=True)
    df['Name'] = df['Name'].str.replace("NPO Details:", "", regex=True)
    df['Reg_no'] = df['Reg_no'].str.replace("Registration No.:", "", regex=True)
    # Remove all the leading spaces
    for column in ['ID', 'Name', 'Reg_no', "cat1", "cat2", "cat3", 'NPO_Type', 'DOR']:
        df[column] = df[column].apply(lambda x:x.strip().replace(r'\s+',' '))
    df["ID"] = df["ID"].astype(int)
    df_final = df.merge(df2, on= ['ID', 'Reg_no', 'DOR'], how='left')
    return df,df_final

In [31]:
#pass in our dfs into 
df,df_final= pre_pocessing(df,df_sample)

In [32]:
df

,ID,Name,Reg_no,NPO_Type,DOR,cat1,cat2,cat3
0,68599106,Reatlegile Day Care Centre,247-285 NPO,Voluntary Association,30/07/2020,Social Services,Services to Children,ECD and Partial Care Centres
1,68539213,Reel Science Co.,246-837 NPO,Not for Profit Companies,16/07/2020,Education and Research,Research,Science and technoloy
2,55790,Regodile Community Development Organisation,055-790 NPO,Voluntary Association,02/07/2007,Health,Hospitals and rehabilitation,Rehabilitation
3,52290592,Restore Alcohol & Drug Recovery Centre Glenvista,216-181 NPO,Not for Profit Companies,02/10/2018,"Law, Advocacy, and Politics",Law and Legal Services,Rehabilitation of offenders
4,34673111,Reya Gona Special Needs Center,185-657 NPO,Voluntary Association,20/02/2017,Social Services,Services to Children,Homes for Children with Special Needs
5,50153123,RHEMA The Cross Ministries,210-934 NPO,Not for Profit Companies,28/06/2018,Religion,Religious Congregations and Associations,Congregations
6,56221573,Rhinosaverz,225-622 NPO,Not for Profit Companies,04/04/2019,Development and Housing,"Economic, Social and Community Development",Social Development
7,43099440,Rise And Shine Youth Skill Development,193-842 NPO,Voluntary Association,04/08/2017,Development and Housing,"Economic, Social and Community Development",Community and neighborhood organisations


In [33]:
df_final

,ID,Name_x,Reg_no,NPO_Type,DOR,cat1,cat2,cat3,Unnamed: 0,Search,Name_y,Province,Status,DC,FM,Page
0,68599106,Reatlegile Day Care Centre,247-285 NPO,Voluntary Association,30/07/2020,Social Services,Services to Children,ECD and Partial Care Centres,0,NaN,Reatlegile Day Care Centre,Gauteng,Registered,,,10
1,68539213,Reel Science Co.,246-837 NPO,Not for Profit Companies,16/07/2020,Education and Research,Research,Science and technoloy,1,NaN,Reel Science Co.,Western Cape,Registered,,,10
2,55790,Regodile Community Development Organisation,055-790 NPO,Voluntary Association,02/07/2007,Health,Hospitals and rehabilitation,Rehabilitation,2,NaN,Regodile Community Development Organisation,Limpopo,Registered,,,10
3,52290592,Restore Alcohol & Drug Recovery Centre Glenvista,216-181 NPO,Not for Profit Companies,02/10/2018,"Law, Advocacy, and Politics",Law and Legal Services,Rehabilitation of offenders,3,NaN,Restore Alcohol & Drug Recovery Centre Glenvista,Gauteng,Registered,,,10
4,34673111,Reya Gona Special Needs Center,185-657 NPO,Voluntary Association,20/02/2017,Social Services,Services to Children,Homes for Children with Special Needs,4,NaN,Reya Gona Special Needs Center,Tembisa,Registered,,,10
5,50153123,RHEMA The Cross Ministries,210-934 NPO,Not for Profit Companies,28/06/2018,Religion,Religious Congregations and Associations,Congregations,5,NaN,RHEMA The Cross Ministries,Gauteng,Registered,,,10
6,56221573,Rhinosaverz,225-622 NPO,Not for Profit Companies,04/04/2019,Development and Housing,"Economic, Social and Community Development",Social Development,6,NaN,Rhinosaverz,Western Cape,Registered,,,10
7,43099440,Rise And Shine Youth Skill Development,193-842 NPO,Voluntary Association,04/08/2017,Development and Housing,"Economic, Social and Community Development",Community and neighborhood organisations,7,NaN,Rise And Shine Youth Skill Development,Hluhluwe,Registered,,,10
